<a href="https://www.kaggle.com/code/mr0106/deep-past-challenge-final?scriptVersionId=301505863" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# -*- coding: utf-8 -*-
"""Deep_Past_Challenge_Final_Fixed.ipynb

Final working solution - CORRECTED
"""

# =============================================================================
# CELL 1: Environment Setup
# =============================================================================

!pip install -q --upgrade pip
!pip install -q torch sacrebleu==2.4.0 transformers sentencepiece \
    pandas numpy tqdm

# Import libraries
import os
import gc
import re
import math
import random
import warnings
import logging
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    NllbTokenizerFast
)

import sacrebleu
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
transformers.logging.set_verbosity_error()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("=" * 60)
print("DEEP PAST CHALLENGE - AKKADIAN TRANSLATION")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("=" * 60)

# =============================================================================
# CELL 2: Configuration
# =============================================================================

@dataclass
class Config:
    """Configuration with CORRECT column names"""
    
    # Data paths
    data_dir: str = '/kaggle/input/competitions/deep-past-initiative-machine-translation'
    output_dir: str = '/kaggle/working'
    submission_file: str = '/kaggle/working/submission.csv'
    
    # Data files
    train_file: str = 'train.csv'
    test_file: str = 'test.csv'
    
    # Column names - IMPORTANT: هذه الأسماء الصحيحة
    train_source: str = 'transliteration'  # اسم عمود الأكدية في train
    train_target: str = 'translation'      # اسم عمود الترجمة في train
    train_id: str = 'oare_id'              # اسم عمود المعرف في train
    
    test_source: str = 'transliteration'   # اسم عمود الأكدية في test - هذا هو الصحيح!
    test_id: str = 'id'                     # اسم عمود المعرف في test
    
    # NLLB model
    model_name: str = 'facebook/nllb-200-distilled-600M'
    src_lang: str = 'akk_Latn'
    tgt_lang: str = 'eng_Latn'
    
    # Data processing
    max_source_length: int = 256
    max_target_length: int = 256
    
    # Inference - optimized for CPU
    batch_size: int = 2
    num_beams: int = 3
    length_penalty: float = 1.0
    early_stopping: bool = True
    no_repeat_ngram_size: int = 3
    
    # System
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers: int = 0
    seed: int = 42

config = Config()

print("\n=== Configuration ===")
for key, value in config.__dict__.items():
    if not key.startswith('_'):
        print(f"{key}: {value}")

# Verify data exists
train_path = os.path.join(config.data_dir, config.train_file)
test_path = os.path.join(config.data_dir, config.test_file)

print(f"\nChecking files:")
print(f"Train exists: {os.path.exists(train_path)}")
print(f"Test exists: {os.path.exists(test_path)}")

# =============================================================================
# CELL 3: Akkadian Preprocessor
# =============================================================================

class AkkadianPreprocessor:
    """
    Clean and normalize Akkadian transliterations
    """
    
    # Character mapping
    CHAR_MAP = {
        'á': 'a', 'à': 'a', 'â': 'a',
        'é': 'e', 'è': 'e', 'ê': 'e',
        'í': 'i', 'ì': 'i', 'î': 'i',
        'ú': 'u', 'ù': 'u', 'û': 'u',
        'š': 'sh', 'Š': 'Sh',
        'ṣ': 's', 'Ṣ': 'S',
        'ṭ': 't', 'Ṭ': 'T',
        'ḫ': 'kh', 'Ḫ': 'Kh',
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
        '{': ' ', '}': ' ',
        '[': ' ', ']': ' ',
        '(': ' ', ')': ' ',
        '<': ' ', '>': ' ',
        '0.3333': ' one-third ',  # Handle common fractions
        '0.5': ' half ',
        '0.25': ' quarter '
    }
    
    def __call__(self, text: str) -> str:
        """Clean Akkadian text"""
        if pd.isna(text) or text is None:
            return '<gap>'
        
        text = str(text)
        
        # Remove scribal notations
        text = re.sub(r'[!?/.:]', ' ', text)
        text = re.sub(r'˹|˺', '', text)
        
        # Handle gaps
        text = re.sub(r'\.\.\.', ' <gap> ', text)
        text = re.sub(r'\[[^\]]*\]', ' <gap> ', text)
        text = re.sub(r'x(?![a-zA-Z])', ' <gap> ', text)
        
        # Map special characters
        for char, repl in self.CHAR_MAP.items():
            text = text.replace(char, repl)
        
        # Clean up multiple spaces
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        return text if text else '<gap>'


class EnglishPreprocessor:
    """Clean English translations"""
    
    def __call__(self, text: str) -> str:
        """Clean English text"""
        if pd.isna(text) or text is None:
            return '<gap>'
        
        text = str(text)
        
        # Basic cleaning
        text = re.sub(r'[!?]', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\s+([.,])', r'\1', text)
        
        # Capitalize first letter
        if text and len(text) > 0:
            text = text[0].upper() + text[1:]
        
        # Ensure proper ending
        text = text.strip()
        if text and text[-1] not in '.!?':
            text += '.'
        
        return text if text else '<gap>'


# =============================================================================
# CELL 4: Load Data with CORRECT Columns
# =============================================================================

def load_data(config):
    """Load train and test data with CORRECT columns"""
    
    print("\n" + "=" * 60)
    print("LOADING DATA")
    print("=" * 60)
    
    # Load train data
    train_path = os.path.join(config.data_dir, config.train_file)
    print(f"Loading train: {train_path}")
    train_df = pd.read_csv(train_path)
    
    print(f"Train samples: {len(train_df)}")
    print(f"Train columns: {train_df.columns.tolist()}")
    
    # Display samples
    print("\nSample Akkadian (train):")
    print(train_df[config.train_source].iloc[0][:200])
    print("\nSample English (train):")
    print(train_df[config.train_target].iloc[0][:200])
    
    # Load test data
    test_path = os.path.join(config.data_dir, config.test_file)
    print(f"\nLoading test: {test_path}")
    test_df = pd.read_csv(test_path)
    
    print(f"Test samples: {len(test_df)}")
    print(f"Test columns: {test_df.columns.tolist()}")
    
    # IMPORTANT: التأكد من استخدام العمود الصحيح
    print("\n" + "=" * 60)
    print("CHECKING TEST COLUMNS")
    print("=" * 60)
    
    # عرض كل الأعمدة للتأكد
    for col in test_df.columns:
        print(f"Column: '{col}'")
        if col == 'transliteration':
            print(f"  ✓ This is the source text column!")
            print(f"  Sample: {test_df[col].iloc[0][:100]}")
        elif col == 'id':
            print(f"  ✓ This is the ID column!")
    
    # تعيين الأعمدة بشكل صريح
    config.test_source = 'transliteration'  # نعم، هذا هو العمود الصحيح
    config.test_id = 'id'
    
    print(f"\n✓ Using source column: '{config.test_source}'")
    print(f"✓ Using ID column: '{config.test_id}'")
    
    return train_df, test_df


# =============================================================================
# CELL 5: Translation Function
# =============================================================================

def translate_texts(model, tokenizer, texts, config):
    """Translate a list of Akkadian texts"""
    
    all_translations = []
    
    # Get target language ID
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(config.tgt_lang)
    
    # Process in batches
    for i in range(0, len(texts), config.batch_size):
        batch_texts = texts[i:i+config.batch_size]
        
        # Skip empty texts
        batch_texts = [t if t and t != '<gap>' else 'missing text' for t in batch_texts]
        
        # Tokenize
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=config.max_source_length
        ).to(config.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=config.max_target_length,
                num_beams=config.num_beams,
                length_penalty=config.length_penalty,
                early_stopping=config.early_stopping,
                no_repeat_ngram_size=config.no_repeat_ngram_size,
                forced_bos_token_id=forced_bos_token_id,
                temperature=0.7,
                do_sample=False
            )
        
        # Decode
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_translations.extend(translations)
        
        # Progress
        if (i // config.batch_size) % 10 == 0:
            print(f"  Processed {min(i + len(batch_texts), len(texts))}/{len(texts)}")
        
        # Clear cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    return all_translations


# =============================================================================
# CELL 6: Quick Test
# =============================================================================

def quick_test(config):
    """Quick test on first training example"""
    
    print("\n" + "=" * 60)
    print("QUICK TEST")
    print("=" * 60)
    
    # Load first training example
    train_path = os.path.join(config.data_dir, config.train_file)
    train_df = pd.read_csv(train_path).head(1)
    
    akk_preprocessor = AkkadianPreprocessor()
    eng_preprocessor = EnglishPreprocessor()
    
    # Load NLLB model
    print(f"\nLoading NLLB model...")
    
    tokenizer = NllbTokenizerFast.from_pretrained(
        config.model_name,
        src_lang=config.src_lang,
        tgt_lang=config.tgt_lang
    )
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to(config.device)
    model.eval()
    
    print(f"✓ Model loaded successfully")
    
    # Test translation
    print(f"\nTesting on 1 example:")
    
    for idx, row in train_df.iterrows():
        akkadian = row[config.train_source]
        reference = row[config.train_target]
        
        # Clean
        cleaned = akk_preprocessor(akkadian)
        
        print(f"\nOriginal Akkadian: {akkadian[:100]}...")
        print(f"Cleaned: {cleaned[:100]}...")
        
        # Translate
        inputs = tokenizer(
            cleaned,
            return_tensors='pt',
            truncation=True,
            max_length=config.max_source_length
        ).to(config.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=config.max_target_length,
                num_beams=config.num_beams,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(config.tgt_lang)
            )
        
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translation = eng_preprocessor(translation)
        
        print(f"Reference: {reference[:100]}...")
        print(f"Translation: {translation[:100]}...")
    
    return True, model, tokenizer


# =============================================================================
# CELL 7: Main Execution
# =============================================================================

def main(config, model=None, tokenizer=None):
    """Main execution function"""
    
    print("\n" + "=" * 70)
    print("DEEP PAST CHALLENGE - TRANSLATION SYSTEM")
    print("=" * 70)
    
    # Load data
    train_df, test_df = load_data(config)
    
    # Initialize preprocessors
    print("\n" + "=" * 60)
    print("INITIALIZING PREPROCESSORS")
    print("=" * 60)
    
    akk_preprocessor = AkkadianPreprocessor()
    eng_preprocessor = EnglishPreprocessor()
    
    # Load model if not provided
    if model is None or tokenizer is None:
        print("\n" + "=" * 60)
        print(f"LOADING NLLB MODEL")
        print("=" * 60)
        
        tokenizer = NllbTokenizerFast.from_pretrained(
            config.model_name,
            src_lang=config.src_lang,
            tgt_lang=config.tgt_lang
        )
        
        model = AutoModelForSeq2SeqLM.from_pretrained(
            config.model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
        model = model.to(config.device)
        model.eval()
        
        print(f"✓ Model loaded successfully")
    
    # Prepare test data
    print("\n" + "=" * 60)
    print("PREPARING TEST DATA")
    print("=" * 60)
    
    # التأكد من استخدام العمود الصحيح - transliteration
    test_texts = test_df[config.test_source].tolist()
    test_ids = test_df[config.test_id].tolist()
    
    print(f"Test texts column: '{config.test_source}'")
    print(f"Test IDs column: '{config.test_id}'")
    print(f"Number of test samples: {len(test_texts)}")
    
    # عرض العينات للتأكد
    print("\nTest samples BEFORE cleaning:")
    for i, text in enumerate(test_texts[:2]):  # عرض أول اثنين فقط
        print(f"  Sample {i}: {str(text)[:100]}...")
    
    # تنظيف النصوص
    print("\nCleaning test texts...")
    cleaned_texts = []
    for i, text in enumerate(tqdm(test_texts, desc="Cleaning")):
        if pd.isna(text) or text is None:
            cleaned = '<gap>'
        else:
            cleaned = akk_preprocessor(str(text))
        cleaned_texts.append(cleaned)
        
        # عرض عينة من أول نص
        if i == 0:
            print(f"\nSample AFTER cleaning:")
            print(f"  Original: {str(text)[:100]}...")
            print(f"  Cleaned: {cleaned[:100]}...")
    
    # Generate translations
    print("\n" + "=" * 60)
    print("GENERATING TRANSLATIONS")
    print("=" * 60)
    print("This may take a few minutes on CPU...")
    
    translations = translate_texts(model, tokenizer, cleaned_texts, config)
    
    # Clean translations
    print("\nCleaning translations...")
    cleaned_translations = []
    for trans in tqdm(translations, desc="Cleaning"):
        cleaned = eng_preprocessor(trans)
        cleaned_translations.append(cleaned)
    
    # Create submission
    print("\n" + "=" * 60)
    print("CREATING SUBMISSION")
    print("=" * 60)
    
    submission_df = pd.DataFrame({
        'id': test_ids,
        'translation': cleaned_translations
    })
    
    # التأكد من التنسيق الصحيح
    submission_df['translation'] = submission_df['translation'].fillna('<gap>')
    submission_df['translation'] = submission_df['translation'].apply(
        lambda x: x if len(x.strip()) > 0 else '<gap>'
    )
    
    # ترتيب حسب id
    submission_df = submission_df.sort_values('id').reset_index(drop=True)
    
    # حفظ الملف
    submission_df.to_csv(config.submission_file, index=False)
    
    print(f"\n✓ Submission saved to: {config.submission_file}")
    print(f"Shape: {submission_df.shape}")
    
    # عرض العينات
    print("\n" + "=" * 60)
    print("TRANSLATION RESULTS:")
    print("=" * 60)
    for i in range(len(submission_df)):
        print(f"ID {submission_df['id'].iloc[i]}:")
        print(f"  Original: {test_texts[i][:100]}...")
        print(f"  Translation: {submission_df['translation'].iloc[i][:100]}...")
        print()
    
    return submission_df


# =============================================================================
# CELL 8: Run
# =============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("STARTING PIPELINE")
    print("=" * 70)
    
    # Quick test first
    test_success, model, tokenizer = quick_test(config)
    
    if test_success:
        print("\n" + "=" * 70)
        print("✓ QUICK TEST PASSED! Running full pipeline...")
        print("=" * 70)
        
        # Run main pipeline
        submission = main(config, model, tokenizer)
        
        print("\n" + "=" * 70)
        print("✓ SUCCESS! Ready for submission")
        print("=" * 70)
        print(f"Submission file: {config.submission_file}")
        
        # Verify file exists and show content
        if os.path.exists(config.submission_file):
            file_size = os.path.getsize(config.submission_file) / 1024
            print(f"File size: {file_size:.2f} KB")
            
            # Show full submission
            print("\n" + "=" * 60)
            print("FINAL SUBMISSION:")
            print("=" * 60)
            final_df = pd.read_csv(config.submission_file)
            print(final_df.to_string())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.8 MB/s eta 0:00:00
DEEP PAST CHALLENGE - AKKADIAN TRANSLATION
PyTorch version: 2.9.0+cpu
Transformers version: 5.2.0
CUDA available: False

=== Configuration ===
data_dir: /kaggle/input/competitions/deep-past-initiative-machine-translation
output_dir: /kaggle/working
submission_file: /kaggle/working/submission.csv
train_file: train.csv
test_file: test.csv
train_source: transliteration
train_target: translation
train_id: oare_id
test_source: transliteration
test_id: id
model_name: facebook/nllb-200-distilled-600M
src_lang: akk_Latn
tgt_lang: eng_Latn
max_source_length: 256
max_target_length: 256
batch_size: 2
num_beams: 3
length_penalty: 1.0
early_stopping: True
no_repeat_ngram_size: 3
device: cpu
num_workers: 0
seed: 42

Checking files:
Train exists: True
Test exists: True

STARTING PIPELINE

QUICK TEST

Loading NLLB model...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

✓ Model loaded successfully

Testing on 1 example:

Original Akkadian: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUM...
Cleaned: KIShIB ma-nu-ba-lum-a-shur DUMU si-la- d IM KIShIB shu- d EN LÍL DUMU ma-nu-ki-a-shur KIShIB MAN-a-s...
Reference: Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur...
Translation: KISHIB ma-nu-ba-lum-a-shur DUMU si-la-d IM KISHIB shu-d EN LIL DUMU ma-nu-ki-a-shur KISHIB MAN-a-shu...

✓ QUICK TEST PASSED! Running full pipeline...

DEEP PAST CHALLENGE - TRANSLATION SYSTEM

LOADING DATA
Loading train: /kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv
Train samples: 1561
Train columns: ['oare_id', 'transliteration', 'translation']

Sample Akkadian (train):
KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.3333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZUR₄-a-šur DUMU a-ta-a a

Cleaning:   0%|          | 0/4 [00:00<?, ?it/s]


Sample AFTER cleaning:
  Original: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-m...
  Cleaned: um-ma ka-ru-um ka-ni-ia-ma a-na aa-qi-il… da-tim ai-ip-ri-ni ka-ar ka-ar-ma u wa-bar-ra-tim qi-bi„-m...

GENERATING TRANSLATIONS
This may take a few minutes on CPU...
  Processed 2/4

Cleaning translations...


Cleaning:   0%|          | 0/4 [00:00<?, ?it/s]


CREATING SUBMISSION

✓ Submission saved to: /kaggle/working/submission.csv
Shape: (4, 2)

TRANSLATION RESULTS:
ID 0:
  Original: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-m...
  Translation: Um-ma ka-ru-um ka-ni-ia-ma a-qi-il... da-tim ai-ip-ri-ni ka-ar ka-r-ma u-bar-ra-tim qi-bi-ma mup-pu-...

ID 1:
  Original: i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú...
  Translation: I'm sure you'll know that I've been there for a while, but I'll be there for you, and I won't let yo...

ID 2:
  Original: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim...
  Translation: I'm going to tell you what I want to do with my life, and I want you to know what I've been doing wi...

ID 3:
  Original: me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-bar-ra-tim aé-bi„-lá KÙ.AN lu a-na DAM.GÀR-ru-tim i-dí-i...
  Translation: Me-e-er mup-pi-ni and a-a